# Phases 3–6: dry-run, probe, bulk load, verification

Follow-along driver for `insert_consents_invest.py` — the script stays the single
source of truth; this notebook imports its functions and walks the run step by step.

Sections 1–2 are local-only. Section 3 authenticates and runs **read-only** SOQL.
Sections 4 (probe: ONE record) and 5 (bulk load) **write to production** and are
each gated behind an explicit flag you have to flip by hand.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root
sys.path.insert(0, str(Path.cwd()))         # this folder: insert_consents_invest

import pandas as pd

import insert_consents_invest as ic
from config import load_mysql_config
from mysql_client import MySQLClient

BATCH_OPTIN = "2026-08-25_invest_central_optin"
BATCH_OPTOUT = "2026-08-25_invest_central_optout"

# Phase 2 contract (staged 2026-08-25, fresh mirror; audit of 08-19 said
# 6,219/11 — population drifted +3, one exclusion match gone, parked by Arsal).
EXPECTED_OPTIN = 6212
EXPECTED_OPTOUT = 10

db = MySQLClient(load_mysql_config())
print("connected | purpose:", ic.CONSENT_NAME, ic.DATA_USE_PURPOSE_ID)

## 1. Load the staged batches (local, read-only)

Same row selection the script uses: flag set, not excluded, CPE present, not yet processed.
Asserted against the phase 2 contract — after a successful load these drop to 0/0, so
flip `AFTER_LOAD = True` for post-load reruns.

In [ ]:
AFTER_LOAD = False  # True once the bulk load ran: open rows are then expected to be 0

def load_batch(batch_id):
    return db.fetch_all(f"""
        SELECT * FROM crm_imp_person_accounts
        WHERE _batch_id = %s AND _excluded = 0
          AND sf_cp_email_id IS NOT NULL
          AND {ic.CONSENT_FLAG_COLUMN} = 1
          AND _consent_processed_at IS NULL
    """, (batch_id,))

rows_optin = load_batch(BATCH_OPTIN)
rows_optout = load_batch(BATCH_OPTOUT)
print(f"OptIn:  {len(rows_optin):,} rows open")
print(f"OptOut: {len(rows_optout):,} rows open")

if not AFTER_LOAD:
    assert len(rows_optin) == EXPECTED_OPTIN, "OptIn batch differs from the phase 2 contract"
    assert len(rows_optout) == EXPECTED_OPTOUT, "OptOut batch differs from the phase 2 contract"
    print("matches the phase 2 contract")

## 2. Build the payload and eyeball it (local)

`row_to_sf_record` is the exact mapper the load uses. Check the field list against D5/D6:
lowercase `Name`, `ConsentKey__c` ending `|CENTRAL`, no Property‑, Hotel‑ or Region fields,
no `CaptureContactPointType`.

In [ ]:
from datetime import datetime, timezone

now = ic.sf_datetime(datetime.now(timezone.utc))
preview = pd.DataFrame([ic.row_to_sf_record(r, "OptIn", now) for r in rows_optin[:5]])
preview

## 3. Authenticate + pre-load duplicate check (prod, READ-ONLY)

`existing_consents` is the same purpose-only SOQL the loader runs before writing.
invest_central has zero consents today, so expect **0 CPEs with existing records** —
after the probe (section 4) exactly the probe CPE shows up here, and after a completed
load this reports everything (which is why reruns are safe).

In [ ]:
from salesforce_client_prod import SalesforceClientCC, load_salesforce_cc_config_from_env

sf = SalesforceClientCC(load_salesforce_cc_config_from_env())
sf.authenticate()
print("authenticated")

found = ic.existing_consents(sf, [str(r["sf_cp_email_id"]) for r in rows_optin + rows_optout])
print(f"CPEs with an existing {ic.CONSENT_NAME} consent: {len(found)}")

## 4. Phase 4 — probe record (prod, WRITES ONE RECORD)

Per the phase doc this must be the **agreed test account**, not an arbitrary row:
put its Account Id into `PROBE_ACCOUNT_ID` (agree it with Oleg/Carmen first), then flip
`RUN_PROBE = True`. The readback afterwards covers the consent, the CPE it hangs on
(ParentId, EmailAddress — to prove it's the right person), and the CPE's consent count.
Business sign-off on the readback **before** section 5.

In [ ]:
RUN_PROBE = False          # <- flip by hand for the one probe insert
PROBE_ACCOUNT_ID = ""      # <- the agreed test account's 18-char Id, from the OptIn batch

if RUN_PROBE:
    candidates = [r for r in rows_optin if r["sf_account_id"] == PROBE_ACCOUNT_ID]
    assert candidates, f"account {PROBE_ACCOUNT_ID!r} is not in the open OptIn batch"
    probe_row = candidates[0]
    payload = ic.row_to_sf_record(probe_row, "OptIn", ic.sf_datetime(datetime.now(timezone.utc)))
    print("payload:", payload)
    r = sf._client.post(f"{sf._base()}/sobjects/ContactPointConsent/", json=payload)
    r.raise_for_status()
    probe_id = r.json()["id"]
    print("probe consent created:", probe_id, "| CPE:", probe_row["sf_cp_email_id"])
else:
    print("probe skipped (RUN_PROBE = False)")

In [ ]:
if RUN_PROBE:
    cpe_id = probe_row["sf_cp_email_id"]

    cpc = sf.query_all(
        f"SELECT Id, Name, ContactPointId, DataUsePurposeId, PrivacyConsentStatus, "
        f"CaptureDate, EffectiveFrom, EffectiveTo, ConsentKey__c, CaptureContactPointType, "
        f"CaptureSource, SourceSystem__c "
        f"FROM ContactPointConsent WHERE Id = '{probe_id}'"
    )["records"]
    cpe = sf.query_all(
        f"SELECT Id, ParentId, EmailAddress FROM ContactPointEmail WHERE Id = '{cpe_id}'"
    )["records"]
    n_consents = sf.query_all(
        f"SELECT COUNT(Id) n FROM ContactPointConsent WHERE ContactPointId = '{cpe_id}'"
    )["records"]

    for rec in cpc + cpe:
        rec.pop("attributes", None)
    print("— consent as Salesforce stored it —")
    display(pd.DataFrame(cpc))
    print("— the CPE it hangs on (verify EmailAddress = the agreed person!) —")
    display(pd.DataFrame(cpe))
    print("consents on this CPE:", n_consents[0]["n"])
    print("check: Name lowercase, key ends |CENTRAL, CaptureContactPointType None")

## 5. Phase 5 — bulk load (prod, WRITES ~6,220 RECORDS)

Runs the script itself, one batch per call, so the real run is exactly what was
dry-run — including the pre-load skip (which now also skips the probe CPE), the
duplicate-CPE abort, the status-conflict detection, and the per-batch
`_consent_processed_at` writeback that makes a crashed run resumable by re-running
the same command. The script exits non-zero on ANY record failure, batch error or
status conflict, which stops this cell before the next batch.

**Does not run without Arsal's explicit go-ahead.** Flip `RUN_LOAD = True` only then.

In [ ]:
import subprocess

RUN_LOAD = False  # <- Arsal's explicit go-ahead required (phase 5 gate)

if RUN_LOAD:
    for batch in (BATCH_OPTIN, BATCH_OPTOUT):
        print(f"\n===== {batch} =====")
        proc = subprocess.run(
            [sys.executable, str(Path.cwd() / "insert_consents_invest.py"), batch],
            cwd=str(Path.cwd().parent), capture_output=True, text=True,
        )
        print(proc.stdout)
        if proc.returncode != 0:
            print(proc.stderr)
            raise RuntimeError(f"{batch} exited {proc.returncode} - fix before continuing")
else:
    print("bulk load skipped (RUN_LOAD = False)")

## 6. Phase 6 — verification (prod, read-only)

Live SOQL counts against the phase 2 contract: 6,212 OptIn (incl. probe) / 10 OptOut,
plus staging writeback completeness (`still_open` must be 0). Any mismatch: check
`<repo-root>/local_data/skipped_*` and `failed_*` before touching anything.

In [ ]:
live = sf.query_all(
    f"SELECT PrivacyConsentStatus, COUNT(Id) n FROM ContactPointConsent "
    f"WHERE DataUsePurposeId = '{ic.DATA_USE_PURPOSE_ID}' GROUP BY PrivacyConsentStatus"
)["records"]
print("live invest_central consents:")
for r in live:
    print(f"  {r['PrivacyConsentStatus']}: {r['n']:,}")

open_rows = db.fetch_df("""
    SELECT _batch_id, SUM(_consent_processed_at IS NULL) AS still_open, COUNT(*) AS total
    FROM crm_imp_person_accounts
    WHERE _batch_id IN (%s, %s)
    GROUP BY _batch_id
""", (BATCH_OPTIN, BATCH_OPTOUT))
print()
print(open_rows.to_string(index=False))